In [1]:
import math


def clean_training_records(records):
    accepted_records = []
    rejected_indices = []

    for i, record in enumerate(records):

        learner_id = record.get("learner_id")
        score = record.get("score")

        # Validate learner_id
        if not isinstance(learner_id, str):
            rejected_indices.append(i)
            continue

        trimmed_learner_id = learner_id.strip()

        if not trimmed_learner_id:
            rejected_indices.append(i)
            continue

        # Reject Boolean scores
        if isinstance(score, bool):
            rejected_indices.append(i)
            continue

        # Accept only int, float, or string
        if not isinstance(score, (int, float, str)):
            rejected_indices.append(i)
            continue

        # Convert score to float
        try:
            float_score = float(score)
        except (ValueError, TypeError, OverflowError):
            rejected_indices.append(i)
            continue

        # Check finite
        if not math.isfinite(float_score):
            rejected_indices.append(i)
            continue

        # Check range
        if not 0 <= float_score <= 100:
            rejected_indices.append(i)
            continue

        # Determine eligibility
        eligible = float_score >= 60

        # Store cleaned record
        accepted_records.append({
            "learner_id": trimmed_learner_id,
            "score": float_score,
            "eligible": eligible
        })

    return accepted_records, rejected_indices

records = [
    {"learner_id": " L01 ", "score": "82.5"},
    {"learner_id": "L02", "score": 60},
    {"learner_id": "L03", "score": None},
    {"learner_id": "   ", "score": 75},
    {"learner_id": "L04", "score": True},
    {"learner_id": "L05", "score": 48.0},
    {"learner_id": "L06", "score": 105},
]
print(clean_training_records(records))

([{'learner_id': 'L01', 'score': 82.5, 'eligible': True}, {'learner_id': 'L02', 'score': 60.0, 'eligible': True}, {'learner_id': 'L05', 'score': 48.0, 'eligible': False}], [2, 3, 4, 6])


In [9]:
def summarise_predictions(results):
    valid_count = 0
    correct_count = 0

    rejected_indices = []
    misclassified_ids = []
    label_counts = {}

    for i, record in enumerate(results):

        if not isinstance(record, dict):
            rejected_indices.append(i)
            continue

        record_id = record.get("id")
        actual = record.get("actual")
        predicted = record.get("predicted")

        if not isinstance(record_id, str) or not record_id.strip():
            rejected_indices.append(i)
            continue

        if not isinstance(actual, str) or not actual.strip():
            rejected_indices.append(i)
            continue

        if not isinstance(predicted, str) or not predicted.strip():
            rejected_indices.append(i)
            continue

        actual = actual.strip().lower()
        predicted = predicted.strip().lower()

        valid_count += 1

        if actual not in label_counts:
            label_counts[actual] = {
                "total": 0,
                "correct": 0
            }

        label_counts[actual]["total"] += 1

        if actual == predicted:
            correct_count += 1
            label_counts[actual]["correct"] += 1
        else:
            misclassified_ids.append(record_id)

    if valid_count > 0:
        accuracy = round(correct_count / valid_count, 4)
    else:
        accuracy = 0.0

    return {
        "valid_count": valid_count,
        "correct_count": correct_count,
        "accuracy": accuracy,
        "label_counts": label_counts,
        "misclassified_ids": misclassified_ids,
        "rejected_indices": rejected_indices
    }


# Input
results = [
    {"id": "T1", "actual": "Billing", "predicted": "billing"},
    {"id": "T2", "actual": "Technical", "predicted": "billing"},
    {"id": "T3", "actual": "billing", "predicted": "billing"},
    {"id": "T4", "actual": "Account", "predicted": "ACCOUNT"},
    {"id": "T5", "actual": None, "predicted": "account"}
]

# IMPORTANT: print the returned result
print(summarise_predictions(results))


{'valid_count': 4, 'correct_count': 3, 'accuracy': 0.75, 'label_counts': {'billing': {'total': 2, 'correct': 2}, 'technical': {'total': 1, 'correct': 0}, 'account': {'total': 1, 'correct': 1}}, 'misclassified_ids': ['T2'], 'rejected_indices': [4]}


In [22]:
from collections import Counter


def audit_dataset_ids(train_ids, validation_ids):

    clean_train = []
    clean_validation = []

    invalid_train_indices = []
    invalid_validation_indices = []

    # -------------------------
    # Clean training IDs
    # -------------------------
    for i, identifier in enumerate(train_ids):

        if not isinstance(identifier, str):
            invalid_train_indices.append(i)
            continue

        identifier = identifier.strip()

        if not identifier:
            invalid_train_indices.append(i)
            continue

        clean_train.append(identifier)

    # -------------------------
    # Clean validation IDs
    # -------------------------
    for i, identifier in enumerate(validation_ids):

        if not isinstance(identifier, str):
            invalid_validation_indices.append(i)
            continue

        identifier = identifier.strip()

        if not identifier:
            invalid_validation_indices.append(i)
            continue

        clean_validation.append(identifier)

    # -------------------------
    # Find duplicates
    # -------------------------
    train_counts = Counter(clean_train)
    validation_counts = Counter(clean_validation)

    train_duplicates = sorted(
        [identifier for identifier, count in train_counts.items()
         if count > 1]
    )

    validation_duplicates = sorted(
        [identifier for identifier, count in validation_counts.items()
         if count > 1]
    )

    # -------------------------
    # Create sets
    # -------------------------
    train_set = set(clean_train)
    validation_set = set(clean_validation)

    # -------------------------
    # Set operations
    # -------------------------
    overlap = sorted(train_set & validation_set)

    train_only = sorted(train_set - validation_set)

    validation_only = sorted(validation_set - train_set)

    # -------------------------
    # Return result
    # -------------------------
    return {
        "train_duplicates":train_duplicates,
        "validation_duplicates": validation_duplicates,
        "overlap": overlap,
        "train_only": train_only,
        "validation_only": validation_only,
        "invalid_train_indices": invalid_train_indices,
        "invalid_validation_indices": invalid_validation_indices,
        "train_snapshot": frozenset(train_set),
        "validation_snapshot": frozenset(validation_set)
    }

train_ids = ["R3", "R1", " R2 ", "R1", None]

validation_ids = ["R2", "R4", "R4", ""]

result = audit_dataset_ids(train_ids, validation_ids)

print(result)

{'train_duplicates': ['R1'], 'validation_duplicates': ['R4'], 'overlap': ['R2'], 'train_only': ['R1', 'R3'], 'validation_only': ['R4'], 'invalid_train_indices': [4], 'invalid_validation_indices': [3], 'train_snapshot': frozenset({'R2', 'R1', 'R3'}), 'validation_snapshot': frozenset({'R4', 'R2'})}


In [26]:
def update_annotation_queue(
    queue,
    incoming,
    completed,
    priority=None,
    batch_size=3
):
    # Combine queue and incoming
    # dict.fromkeys() removes duplicates
    # while preserving first occurrence
    queue_up = list(dict.fromkeys(queue + incoming))

    # Remove completed IDs
    completed_set = set(completed)

    queue_up = [
        document_id
        for document_id in queue_up
        if document_id not in completed_set
    ]

    # Move priority to the front
    if priority is not None and priority in queue_up:
        queue_up.remove(priority)
        queue_up.insert(0, priority)

    # First batch using slicing
    next_batch = queue_up[:batch_size]

    # Final item using indexing
    if queue_up:
        final_item = queue_up[len(queue_up) - 1]
    else:
        final_item = None

    # Numbered positions using range
    positions = []

    for i in range(len(queue_up)):
        positions.append((i + 1, queue_up[i]))

    return queue_up, next_batch, final_item, positions

queue = ["D3", "D1", "D3", "D2"]
incoming = ["D4", "D2", "D5"]
completed = ["D1", "D9"]
update_annotation_queue(
    queue, incoming, completed,
    priority="D5", batch_size=3,
)
result=update_annotation_queue(
    queue,
    incoming,
    completed,
    priority=None,
    batch_size=3
)
print(result)

(['D3', 'D2', 'D4', 'D5'], ['D3', 'D2', 'D4'], 'D5', [(1, 'D3'), (2, 'D2'), (3, 'D4'), (4, 'D5')])


In [27]:
def update_annotation_queue(
    queue,
    incoming,
    completed,
    priority=None,
    batch_size=3
):
    # Combine queue and incoming, remove duplicates
    queue_up = list(dict.fromkeys(queue + incoming))

    # Remove completed IDs
    completed_set = set(completed)

    queue_up = [
        document_id
        for document_id in queue_up
        if document_id not in completed_set
    ]

    # Move priority to the front
    if priority is not None and priority in queue_up:
        queue_up.remove(priority)
        queue_up.insert(0, priority)

    # First batch
    next_batch = queue_up[:batch_size]

    # Final item
    if queue_up:
        final_item = queue_up[len(queue_up) - 1]
    else:
        final_item = None

    # Numbered positions
    positions = []

    for i in range(len(queue_up)):
        positions.append((i + 1, queue_up[i]))

    return queue_up, next_batch, final_item, positions

In [28]:
queue = ["D3", "D1", "D3", "D2"]

incoming = ["D4", "D2", "D5"]

completed = ["D1", "D9"]

result = update_annotation_queue(
    queue,
    incoming,
    completed,
    priority="D5",
    batch_size=3
)

print(result)

(['D5', 'D3', 'D2', 'D4'], ['D5', 'D3', 'D2'], 'D4', [(1, 'D5'), (2, 'D3'), (3, 'D2'), (4, 'D4')])


In [6]:
from copy import deepcopy
from typing import Any


def build_experiment_config(
    base: dict[str, Any],
    *,
    run_name: str,
    parameter_updates: dict[str, Any] | None = None,
    **metadata: Any
) -> dict[str, Any]:

    # Create an independent deep copy
    config = deepcopy(base)

    # Add trimmed run name
    config["run_name"] = run_name.strip()

    # Apply parameter updates
    if parameter_updates is not None:
        for key, value in parameter_updates.items():

            # Check whether parameter already exists
            if key not in config["parameters"]:
                raise KeyError(key)

            config["parameters"][key] = value

    # Merge additional metadata
    config["metadata"].update(metadata)

    return config


# -------------------------
# Sample input
# -------------------------

base = {
    "model": "ticket_classifier",
    "parameters": {
        "threshold": 0.5,
        "max_tokens": 128
    },
    "metadata": {
        "owner": "platform",
        "stage": "baseline"
    }
}


# Call the function
result = build_experiment_config(
    base,
    run_name="trial_02",
    parameter_updates={"threshold": 0.7},
    owner="nlp_team",
    seed=42
)


# Print the result
print("New configuration:")
print(result)

print("\nOriginal base configuration:")
print(base)

New configuration:
{'model': 'ticket_classifier', 'parameters': {'threshold': 0.7, 'max_tokens': 128}, 'metadata': {'owner': 'nlp_team', 'stage': 'baseline', 'seed': 42}, 'run_name': 'trial_02'}

Original base configuration:
{'model': 'ticket_classifier', 'parameters': {'threshold': 0.5, 'max_tokens': 128}, 'metadata': {'owner': 'platform', 'stage': 'baseline'}}


In [8]:
import math


def aggregate_scores(*batches, top_n=3):

    valid_scores = []
    rejected_positions = []

    for batch_index, batch in enumerate(batches):
        if not isinstance(batch, (list, tuple)):
            raise TypeError("Each batch must be a list or tuple")

        for item_index, score in enumerate(batch):

            if (
                isinstance(score, bool)
                or not isinstance(score, (int, float))
                or not math.isfinite(score)
                or not 0 <= score <= 100
            ):
                rejected_positions.append(
                    (batch_index, item_index)
                )
            else:
                valid_scores.append(
                    (batch_index, item_index, float(score))
                )

    valid_count = len(valid_scores)

    mean = round(
        sum(score for _, _, score in valid_scores) / valid_count,
        2
    ) if valid_count else 0.0

    valid_scores.sort(
        key=lambda x: (-x[2], x[0], x[1])
    )

    return (
        valid_count,
        mean,
        valid_scores[:top_n],
        rejected_positions
    )


# Test
batch_a = [80, 90, None]
batch_b = (90, 70, True)
batch_c = []

result = aggregate_scores(
    batch_a,
    batch_b,
    batch_c,
    top_n=3
)

print("Valid count:", result[0])
print("Mean:", result[1])
print("Top scores:", result[2])
print("Rejected positions:", result[3])

Valid count: 4
Mean: 82.5
Top scores: [(0, 1, 90.0), (1, 0, 90.0), (0, 0, 80.0)]
Rejected positions: [(0, 2), (1, 2)]


In [1]:
def summarise_tool_calls(records):

    # --------------------------------
    # 1. Keep only the last record
    #    for each call_id
    # --------------------------------

    latest = {}

    for record in records:
        latest[record["call_id"]] = record


    # --------------------------------
    # 2. Find all tools
    # --------------------------------

    tools = {record["tool"] for record in latest.values()}


    # --------------------------------
    # 3. Dictionary comprehension
    #    to create the initial report
    # --------------------------------

    report = {
        tool: {
            "total_calls": 0,
            "successes": 0,
            "failures": 0,
            "total_duration_ms": 0
        }
        for tool in tools
    }


    # --------------------------------
    # 4. Calculate statistics
    # --------------------------------

    failed_call_ids = []

    for record in latest.values():

        tool = record["tool"]
        status = record["status"]
        duration = record["duration_ms"]

        # Total calls
        report[tool]["total_calls"] += 1

        # Success / failure
        if status == "ok":
            report[tool]["successes"] += 1

        elif status == "error":
            report[tool]["failures"] += 1
            failed_call_ids.append(record["call_id"])

        # Duration
        report[tool]["total_duration_ms"] += duration


    # --------------------------------
    # 5. Sort failed call IDs
    # --------------------------------

    failed_call_ids.sort()


    # --------------------------------
    # 6. Sort tools
    #
    # failure count → descending
    # total calls   → descending
    # tool name     → ascending
    # --------------------------------

    report = dict(
        sorted(
            report.items(),
            key=lambda item: (
                -item[1]["failures"],
                -item[1]["total_calls"],
                item[0]
            )
        )
    )


    # --------------------------------
    # 7. Return final report
    # --------------------------------

    return {
        "tools": report,
        "failed_call_ids": failed_call_ids
    }


# =================================
# Sample input
# =================================

records = [
    {
        "call_id": "C1",
        "tool": "search",
        "status": "error",
        "duration_ms": 120
    },
    {
        "call_id": "C2",
        "tool": "calculator",
        "status": "ok",
        "duration_ms": 10
    },
    {
        "call_id": "C1",
        "tool": "search",
        "status": "ok",
        "duration_ms": 80
    },
    {
        "call_id": "C3",
        "tool": "search",
        "status": "error",
        "duration_ms": 40
    },
    {
        "call_id": "C4",
        "tool": "retriever",
        "status": "error",
        "duration_ms": 50
    }
]


# =================================
# Call function
# =================================

result = summarise_tool_calls(records)


# =================================
# Print result
# =================================

print(result)

{'tools': {'search': {'total_calls': 2, 'successes': 1, 'failures': 1, 'total_duration_ms': 120}, 'retriever': {'total_calls': 1, 'successes': 0, 'failures': 1, 'total_duration_ms': 50}, 'calculator': {'total_calls': 1, 'successes': 1, 'failures': 0, 'total_duration_ms': 10}}, 'failed_call_ids': ['C3', 'C4']}


In [5]:
from functools import wraps
from typing import Callable


def validate_text_batch(func: Callable) -> Callable:
    @wraps(func)
    def wrapper(*args, **kwargs):

        # Find texts from positional or keyword argument
        if args:
            texts = args[0]
        elif "texts" in kwargs:
            texts = kwargs["texts"]
        else:
            raise TypeError("Missing required argument: 'texts'")

        # texts must be a list
        if not isinstance(texts, list):
            raise TypeError("texts must be a list")

        # Every item must be a string and not blank
        for text in texts:
            if not isinstance(text, str):
                raise TypeError("Every item in texts must be a string")

            if text.strip() == "":
                raise ValueError(
                    "texts cannot contain empty or whitespace-only strings"
                )

        # Run the original function
        return func(*args, **kwargs)

    return wrapper


@validate_text_batch
def text_lengths(
    texts: list[str],
    *,
    strip: bool = True
) -> list[int]:

    if strip:
        return [len(text.strip()) for text in texts]

    return [len(text) for text in texts]


# Test
texts = ["  agent ", "ML", "data"]

print(text_lengths(texts))
print(text_lengths(texts=texts, strip=False))




[5, 2, 4]
[8, 2, 4]


In [9]:
from functools import wraps


def track_calls(history):

    def decorator(func):

        @wraps(func)
        def wrapper(*args, **kwargs):

            call_number = len(history) + 1

            try:
                result = func(*args, **kwargs)

                history.append({
                    "call_number": call_number,
                    "function": func.__name__,
                    "status": "success",
                    "error_type": None
                })

                return result

            except Exception as error:

                history.append({
                    "call_number": call_number,
                    "function": func.__name__,
                    "status": "error",
                    "error_type": type(error).__name__
                })

                raise

        return wrapper

    return decorator


# IMPORTANT: Create history BEFORE using @track_calls(history)
history = []


@track_calls(history)
def average_score(scores: list[int | float]) -> float:

    if len(scores) == 0:
        raise ValueError("scores cannot be empty")

    return sum(scores) / len(scores)


@track_calls(history)
def format_run_name(prefix: str, *, number: int) -> str:

    return f"{prefix}_{number:02d}"


# Test
print(average_score([60, 80]))
print(format_run_name("eval", number=3))

try:
    average_score([])
except ValueError:
    pass

print(history)

70.0
eval_03
[{'call_number': 1, 'function': 'average_score', 'status': 'success', 'error_type': None}, {'call_number': 2, 'function': 'format_run_name', 'status': 'success', 'error_type': None}, {'call_number': 3, 'function': 'average_score', 'status': 'error', 'error_type': 'ValueError'}]
